# 📦 준비된 데이터셋 번들 불러오기

이 노트북은 사전에 생성·검증된 **τ-Knowledge banking_knowledge** 학습 데이터 번들을
가져오고, 무결성을 검증하며, 내용을 살펴봅니다.

## 데이터-퍼스트 원칙

- 학습자 노트북에서는 **SDG(합성 데이터 생성)을 실행하지 않습니다**.
- 운영자/고급 사용자가 미리 생성한 번들을 다운로드하여 사용합니다.
- 번들이 유효하지 않으면 학습을 시작하지 않고, 저작 경로(authoring path)로 안내합니다.

## 번들 구조

```
tau-knowledge-v1/
├── manifest.json          # 번들 메타데이터
├── checksums.sha256       # 무결성 체크섬
├── canonical/             # 정규 학습/검증 데이터
├── training/lora/         # LoRA 백엔드 전용 포맷
├── training/osft/         # OSFT 백엔드 전용 포맷
├── kb/                    # KB(지식 기반) 스냅샷
├── metadata/              # 출처·분할 메타데이터
└── reports/               # 품질 리포트
```

In [ ]:
"""환경 확인 — 00_preflight.ipynb 에서 이미 설치 완료."""

import os
from pathlib import Path

_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

os.environ["RHOAI_PROJECT_ROOT"] = str(_project_root)

try:
    import rhoai_model_training_lab.config as _cfg
    _cfg.PROJECT_ROOT = _project_root
    print(f"✅ 프로젝트 루트: {_project_root}")
except ImportError:
    raise ImportError(
        "❌ 패키지 미설치 — 먼저 00_preflight.ipynb 를 실행하세요."
    )


In [ ]:
"""Fetch the prepared dataset bundle."""

import os
import subprocess
import sys
from pathlib import Path

from rhoai_model_training_lab.config import load_env, load_bundle_config, PROJECT_ROOT

load_env()

release_config = load_bundle_config()
bundle_cfg = release_config.get("bundle", {})
bundle_name = f"{bundle_cfg.get('name', 'tau-knowledge')}-{bundle_cfg.get('version', 'v1')}"
default_bundle_path = Path(bundle_cfg.get("base_path", f"data/prepared/{bundle_name}"))

if not default_bundle_path.is_absolute():
    default_bundle_path = PROJECT_ROOT / default_bundle_path

print(f"번들 이름: {bundle_name}")
print(f"기본 경로: {default_bundle_path}")

# Try fetching via the fetch script
fetch_script = PROJECT_ROOT / "scripts" / "fetch_prepared_dataset.py"

if default_bundle_path.exists() and (default_bundle_path / "manifest.json").exists():
    print(f"\n✅ 번들이 이미 존재합니다: {default_bundle_path}")
    bundle_path = default_bundle_path
elif fetch_script.exists():
    print(f"\n📥 번들 다운로드 중...")
    result = subprocess.run(
        [sys.executable, str(fetch_script), "--release", "configs/data-release.yaml"],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        print("✅ 번들 다운로드 완료")
        bundle_path = default_bundle_path
    else:
        print(f"❌ 다운로드 실패:\n{result.stderr}")
        raise RuntimeError("번들을 가져올 수 없습니다. 네트워크 및 S3/PVC 설정을 확인하세요.")
else:
    # Try the data module's fetch_bundle
    from rhoai_model_training_lab.data import fetch_bundle
    bundle_path = fetch_bundle(release_config)
    print(f"✅ 번들 로드 완료: {bundle_path}")

In [ ]:
"""Validate checksums and compatibility."""

from rhoai_model_training_lab.data import BundleManager, validate_prepared_dataset

# Comprehensive validation
print("📋 번들 무결성 검증 중...\n")
validation = validate_prepared_dataset(bundle_path)

if validation["valid"]:
    print("✅ 번들 검증 통과")
else:
    print("❌ 번들 검증 실패")
    for err in validation["errors"]:
        print(f"  ❌ {err}")

if validation["warnings"]:
    print("\n⚠️  경고:")
    for warn in validation["warnings"]:
        print(f"  ⚠️  {warn}")

print("\n요약:")
for k, v in validation.get("summary", {}).items():
    print(f"  {k}: {v}")

if not validation["valid"]:
    raise RuntimeError(
        "번들 검증 실패 — 학습을 진행할 수 없습니다.\n"
        "저작 경로(data_preparation/)를 통해 번들을 재생성하세요."
    )

In [ ]:
"""Load and display the bundle manifest."""

import json
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Display manifest
table = Table(title="📄 번들 매니페스트", show_header=True)
table.add_column("항목", style="bold cyan")
table.add_column("값")

manifest_items = [
    ("번들 이름", manifest.bundle_name),
    ("버전", manifest.bundle_version),
    ("생성일", manifest.created_at),
    ("τ 버전", manifest.tau_version),
    ("τ 커밋 SHA", manifest.tau_commit_sha[:12] + "..." if manifest.tau_commit_sha else "N/A"),
    ("모델 ID", manifest.model_id),
    ("모델 리비전", manifest.model_revision),
    ("토크나이저 ID", manifest.tokenizer_id),
    ("학습 샘플 수", str(manifest.canonical_train_count)),
    ("검증 샘플 수", str(manifest.canonical_validation_count)),
    ("분할 정책", manifest.split_policy),
]

for label, value in manifest_items:
    table.add_row(label, value)

console.print(table)

# Type distribution
if manifest.sample_type_distribution:
    print("\n📊 샘플 유형 분포:")
    for stype, count in sorted(manifest.sample_type_distribution.items()):
        total = manifest.canonical_train_count + manifest.canonical_validation_count
        pct = (count / total * 100) if total > 0 else 0
        bar = "█" * int(pct / 2)
        print(f"  {stype:<25} {count:>5} ({pct:5.1f}%) {bar}")

In [ ]:
"""Show sample statistics and type distribution."""

from collections import Counter

# Load canonical samples
train_samples = mgr.get_samples("train")
val_samples = mgr.get_samples("validation")

print(f"학습 샘플 수: {len(train_samples)}")
print(f"검증 샘플 수: {len(val_samples)}")
print(f"총 샘플 수: {len(train_samples) + len(val_samples)}")

# Type distribution breakdown
print("\n--- 학습 데이터 유형별 통계 ---")
train_types = Counter(s.sample_type for s in train_samples)
for stype, count in train_types.most_common():
    pct = count / len(train_samples) * 100
    print(f"  {stype.value:<25} {count:>5} ({pct:.1f}%)")

print("\n--- 검증 데이터 유형별 통계 ---")
val_types = Counter(s.sample_type for s in val_samples)
for stype, count in val_types.most_common():
    pct = count / len(val_samples) * 100
    print(f"  {stype.value:<25} {count:>5} ({pct:.1f}%)")

# Message length statistics
print("\n--- 메시지 길이 통계 ---")
msg_counts = [len(s.messages) for s in train_samples]
print(f"  평균 메시지 수: {sum(msg_counts)/len(msg_counts):.1f}")
print(f"  최소/최대: {min(msg_counts)} / {max(msg_counts)}")

# Tool usage stats
samples_with_tools = sum(1 for s in train_samples if s.tools)
print(f"\n  도구 스키마 포함 샘플: {samples_with_tools} ({samples_with_tools/len(train_samples)*100:.1f}%)")

# Validation status
status_counts = Counter(s.validation_status for s in train_samples)
print("\n--- 검증 상태 ---")
for status, count in status_counts.most_common():
    print(f"  {status.value:<20} {count:>5}")

## 📊 데이터셋 EDA (Exploratory Data Analysis)

학습 데이터의 분포와 특성을 시각적으로 탐색합니다.

In [ ]:
"""데이터셋 EDA — 분포, 길이, 구성 시각화."""

try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("ℹ️  matplotlib 미설치 — 텍스트 요약만 출력합니다.")

from collections import Counter

all_samples = train_samples + val_samples

# ── 1. 유형별 분포 (train vs val) ──
train_types = Counter(s.sample_type.value for s in train_samples)
val_types = Counter(s.sample_type.value for s in val_samples)
all_type_keys = sorted(set(train_types) | set(val_types))

if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("📊 데이터셋 EDA", fontsize=15, fontweight="bold")

    # 1-1. 유형별 학습/검증 분포
    ax = axes[0, 0]
    x = range(len(all_type_keys))
    w = 0.35
    ax.bar([i - w/2 for i in x], [train_types.get(k, 0) for k in all_type_keys], w, label="학습", color="#4C72B0")
    ax.bar([i + w/2 for i in x], [val_types.get(k, 0) for k in all_type_keys], w, label="검증", color="#DD8452")
    ax.set_xticks(list(x))
    ax.set_xticklabels(all_type_keys, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("샘플 수")
    ax.set_title("유형별 학습/검증 분포")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # 1-2. 메시지 수(턴) 분포
    ax = axes[0, 1]
    train_msg_lens = [len(s.messages) for s in train_samples]
    val_msg_lens = [len(s.messages) for s in val_samples]
    ax.hist(train_msg_lens, bins=range(1, max(train_msg_lens + val_msg_lens) + 2),
            alpha=0.7, label="학습", color="#4C72B0", edgecolor="white")
    ax.hist(val_msg_lens, bins=range(1, max(train_msg_lens + val_msg_lens) + 2),
            alpha=0.7, label="검증", color="#DD8452", edgecolor="white")
    ax.set_xlabel("메시지 수 (턴)")
    ax.set_ylabel("샘플 수")
    ax.set_title("대화 길이 분포")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # 1-3. 토큰 길이 추정 (문자 수 기반)
    ax = axes[1, 0]
    def total_chars(sample):
        return sum(len(m.content or "") for m in sample.messages)
    train_chars = [total_chars(s) for s in train_samples]
    val_chars = [total_chars(s) for s in val_samples]
    ax.hist(train_chars, bins=30, alpha=0.7, label="학습", color="#4C72B0", edgecolor="white")
    ax.hist(val_chars, bins=30, alpha=0.7, label="검증", color="#DD8452", edgecolor="white")
    ax.set_xlabel("총 문자 수")
    ax.set_ylabel("샘플 수")
    ax.set_title("샘플별 총 문자 수 분포")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else f"{x:.0f}"))

    # 1-4. 역할(role)별 메시지 비율
    ax = axes[1, 1]
    role_counts = Counter()
    for s in all_samples:
        for m in s.messages:
            role_counts[m.role] += 1
    roles = sorted(role_counts.keys())
    counts = [role_counts[r] for r in roles]
    colors = {"system": "#55A868", "user": "#4C72B0", "assistant": "#DD8452", "tool": "#C44E52"}
    ax.pie(counts, labels=roles, autopct="%1.1f%%",
           colors=[colors.get(r, "#999999") for r in roles],
           startangle=90)
    ax.set_title("역할별 메시지 비율")

    plt.tight_layout()
    plt.show()

# ── 2. 텍스트 요약 (항상 출력) ──
print("\n" + "=" * 60)
print("📋 EDA 요약")
print("=" * 60)

print(f"\n총 샘플: {len(all_samples)} (학습 {len(train_samples)} / 검증 {len(val_samples)})")
print(f"학습:검증 비율 = {len(train_samples)/len(all_samples)*100:.0f}:{len(val_samples)/len(all_samples)*100:.0f}")

print(f"\n유형별 분포:")
for k in all_type_keys:
    tr, va = train_types.get(k, 0), val_types.get(k, 0)
    print(f"  {k:<25} 학습 {tr:>4}  검증 {va:>4}  합계 {tr+va:>4}")

all_msg_lens = [len(s.messages) for s in all_samples]
print(f"\n대화 길이 (메시지 수):")
print(f"  평균: {sum(all_msg_lens)/len(all_msg_lens):.1f}  중앙값: {sorted(all_msg_lens)[len(all_msg_lens)//2]}")
print(f"  최소: {min(all_msg_lens)}  최대: {max(all_msg_lens)}")

all_chars = [sum(len(m.content or "") for m in s.messages) for s in all_samples]
print(f"\n문자 수 (토큰 추정 ≈ 문자수/3.5):")
print(f"  평균: {sum(all_chars)/len(all_chars):,.0f} 자 (~{sum(all_chars)/len(all_chars)/3.5:,.0f} 토큰)")
print(f"  최소: {min(all_chars):,} 자  최대: {max(all_chars):,} 자")

tools_count = sum(1 for s in all_samples if s.tools)
print(f"\n도구 사용: {tools_count}/{len(all_samples)} ({tools_count/len(all_samples)*100:.1f}%)")

families = Counter(s.scenario_family for s in all_samples if s.scenario_family)
if families:
    print(f"\n시나리오 패밀리: {len(families)}개")
    for fam, cnt in families.most_common(10):
        print(f"  {fam:<30} {cnt:>4}")
    if len(families) > 10:
        print(f"  ... 외 {len(families)-10}개")


In [ ]:
"""Preview example samples (one per type)."""

from rich.syntax import Syntax

print("=" * 70)
print("📝 유형별 샘플 미리보기 (학습 데이터에서 각 유형 1개씩)")
print("=" * 70)

seen_types = set()
for sample in train_samples:
    if sample.sample_type in seen_types:
        continue
    seen_types.add(sample.sample_type)

    print(f"\n{'─' * 70}")
    print(f"유형: {sample.sample_type.value}")
    print(f"ID: {sample.sample_id}")
    if sample.source_doc_ids:
        print(f"출처 문서: {', '.join(sample.source_doc_ids[:3])}")
    if sample.scenario_family:
        print(f"시나리오 패밀리: {sample.scenario_family}")
    print(f"메시지 수: {len(sample.messages)}")
    if sample.tools:
        tool_names = [t.function.name for t in sample.tools]
        print(f"도구: {', '.join(tool_names)}")
    print()

    # Show messages (truncated)
    for msg in sample.messages:
        role_icon = {"system": "🔧", "user": "👤", "assistant": "🤖", "tool": "🔨"}.get(msg.role, "❓")
        content = msg.content or ""
        if len(content) > 300:
            content = content[:300] + "..."
        print(f"  {role_icon} [{msg.role}]: {content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"     🔧 tool_call: {tc.function.name}({tc.function.arguments[:100]})")

print(f"\n{'=' * 70}")
print(f"총 {len(seen_types)}개 유형의 샘플을 미리보기했습니다.")

In [ ]:
"""Print validation summary."""

table = Table(title="✅ 데이터셋 번들 검증 요약", show_header=True)
table.add_column("검증 항목", style="bold")
table.add_column("결과")

summary_items = [
    ("번들 경로", str(bundle_path)),
    ("매니페스트 유효", "✅"),
    ("체크섬 검증", "✅" if validation.get("summary", {}).get("checksums_ok") else "❌"),
    ("학습 샘플", f"{len(train_samples):,}"),
    ("검증 샘플", f"{len(val_samples):,}"),
    ("유형 수", str(len(seen_types))),
    ("도구 포함 샘플", f"{samples_with_tools:,}"),
    ("대상 모델", manifest.model_id),
    ("번들 상태", "✅ 학습 준비 완료" if validation["valid"] else "❌ 검증 실패"),
]

for label, value in summary_items:
    table.add_row(label, value)

console.print(table)

print("\n다음 단계:")
print("  📓 03_lora_finetuning.ipynb  — LoRA 파인튜닝 실행")
print("  📓 04_osft_finetuning.ipynb  — OSFT 파인튜닝 실행")
print("  두 학습은 동일한 기본 모델과 데이터에서 독립적으로 진행됩니다.")